<a href="https://colab.research.google.com/github/Nandishsk-07/DarkLingo/blob/main/DarkLingo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!nvidia-smi
!pip install -q transformers scikit-learn pandas nltk torch

Thu Nov 26 13:29:09 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   48C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
import os, zipfile, nltk

# 1. Unzip your uploaded archive
assert os.path.exists("darklingo_paper.zip"), "Upload darklingo_paper.zip to /content first!"
with zipfile.ZipFile("darklingo_paper.zip", "r") as z:
    z.extractall(".")

# 2. Download NLTK stopwords required by real_eval.py
nltk.download("stopwords", quiet=True)

# 3. Clone Yada et al. real dataset repository
if not os.path.exists("ec-darkpattern"):
    !git clone -q https://github.com/yamanalab/ec-darkpattern.git

# 4. Copy scripts and data to working directory so imports work cleanly
!cp darklingo_paper/scripts/real_eval.py .
!cp darklingo_paper/scripts/finetune_distilbert.py .
!cp darklingo_paper/data/synthetic.csv .

print("Setup complete. Files ready:", os.path.exists("synthetic.csv"), os.path.exists("ec-darkpattern/dataset/dataset.tsv"))

Setup complete. Files ready: True True


In [3]:
with open("finetune_distilbert.py", "r") as f:
    code = f.read()

code = code.replace(
    "from sklearn.metrics import accuracy_score, f1_score",
    "from sklearn.metrics import accuracy_score, f1_score, precision_recall_fscore_support"
)

old_append = "rows.append(dict(acc=accuracy_score(y[te], p), mF1=f1_score(y[te], p, average=\"macro\"), train_s=ts, infer_s=inf))"
new_append = (
    "mp, mr, mf1, _ = precision_recall_fscore_support(y[te], p, average=\"macro\", zero_division=0)\n"
    "        rows.append(dict(acc=accuracy_score(y[te], p), mP=mp, mR=mr, mF1=mf1, train_s=ts, infer_s=inf))"
)
assert old_append in code, "Could not find target line to patch"
code = code.replace(old_append, new_append)

with open("finetune_distilbert.py", "w") as f:
    f.write(code)

print("Patched finetune_distilbert.py to record mP and mR alongside acc, mF1, train_s, infer_s.")

Patched finetune_distilbert.py to record mP and mR alongside acc, mF1, train_s, infer_s.


In [4]:
# 1. Real data (4-class, page_id grouped, 5 folds x 3 seeds = 15 folds) -> Table VI
!python finetune_distilbert.py --real ec-darkpattern/dataset/dataset.tsv --out real_distilbert.json

# 2. Synthetic data (6-class, frame_id grouped, 5 folds x 3 seeds = 15 folds) -> Table V
!python finetune_distilbert.py --csv synthetic.csv --group frame_id --out synth_distilbert.json

config.json: 100% 483/483 [00:00<00:00, 1.87MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 225kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 9.19MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 4.37MB/s]

model.safetensors: downloading bytes:  69% 184M/268M [00:00<00:00, 292MB/s, 13.5MB/s  ]
model.safetensors: downloading bytes:  86% 230M/268M [00:01<00:00, 264MB/s, 20.3MB/s  ]
model.safetensors: downloading bytes: 100% 250M/250M [00:01<00:00, 160MB/s, 23.6MB/s  ]
model.safetensors: reconstructing file: 100% 268M/268M [00:01<00:00, 171MB/s, 25.4MB/s  ]
Loading weights: 100% 100/100 [00:00<00:00, 7442.38it/s]
[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 

In [5]:
import json, subprocess

gpu_name = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]
).decode().strip().splitlines()[0]

syn = json.load(open("synth_distilbert.json"))
real = json.load(open("real_distilbert.json"))

def fmt(d, k):
    # Formats as .XXX \pm .XXX matching IEEE tables in main.tex
    m = f"{d['mean'][k]:.3f}".lstrip("0")
    s = f"{d['std'][k]:.3f}".lstrip("0")
    return f"{m} \\pm {s}"

print("=========================================================")
print(f"GPU: {gpu_name}")
print("---------------------------------------------------------")
print("TABLE V (Synthetic, frame-grouped, 15 folds):")
print(f"DistilBERT (pretrained) & {fmt(syn, 'acc')} & {fmt(syn, 'mF1')} \\\\")
print(f"Timing (Synthetic): train_s = {syn['mean']['train_s']:.2f}s/fold, infer_s = {syn['mean']['infer_s']*1000:.3f} ms/sentence")
print("---------------------------------------------------------")
print("TABLE VI (Real 4-class, page-grouped, 15 folds):")
print(f"DistilBERT (pretrained) & {fmt(real, 'acc')} & {fmt(real, 'mP')} & {fmt(real, 'mR')} & {fmt(real, 'mF1')} \\\\")
print(f"Timing (Real): train_s = {real['mean']['train_s']:.2f}s/fold, infer_s = {real['mean']['infer_s']*1000:.3f} ms/sentence")
print("=========================================================")

GPU: Tesla T4
---------------------------------------------------------
TABLE V (Synthetic, frame-grouped, 15 folds):
DistilBERT (pretrained) & .900 \pm .065 & .897 \pm .067 \\
Timing (Synthetic): train_s = 23.93s/fold, infer_s = 0.723 ms/sentence
---------------------------------------------------------
TABLE VI (Real 4-class, page-grouped, 15 folds):
DistilBERT (pretrained) & .984 \pm .007 & .980 \pm .011 & .981 \pm .009 & .980 \pm .009 \\
Timing (Real): train_s = 24.56s/fold, infer_s = 1.687 ms/sentence
